In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
DATASET_PATH = "dataset"
print("Dataset var mı?", os.path.exists(DATASET_PATH))

In [ ]:
print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU bulundu:")
    for gpu in gpus:
        print(gpu)

    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth aktif edildi.")
    except RuntimeError as e:
        print(e)

else:
    print("GPU bulunamadı. Model CPU ile çalışacak.")

In [ ]:
devices = tf.config.list_physical_devices()
print("Cihazlar:", devices)

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"Harika! {len(gpu_devices)} adet GPU aktif.")
else:
    print("GPU bulunamadı, kurulumu kontrol etmelisin.")

In [ ]:
for folder in os.listdir(DATASET_PATH):
    folder_path = os.path.join(DATASET_PATH, folder)
    
    if os.path.isdir(folder_path):
        print(folder)

In [ ]:
image_extensions = [".jpg", ".jpeg", ".png", ".webp"]

data = []

for img_path in Path(DATASET_PATH).rglob("*"):
    if img_path.suffix.lower() in image_extensions:
        label = img_path.parent.name
        
        data.append({
            "filepath": str(img_path),
            "label": label
        })

df = pd.DataFrame(data)

print(df.head())
print(df["label"].value_counts())
print("Toplam görsel sayısı:", len(df))
print("Toplam class sayısı:", df["label"].nunique())

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, x="label", order=df["label"].value_counts().index)
plt.xticks(rotation=45)
plt.title("Class Dağılımı")
plt.xlabel("Class")
plt.ylabel("Görsel Sayısı")
plt.show()

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)
val_test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_gen = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print(train_gen.class_indices)

In [ ]:
num_classes = len(train_gen.class_indices)
print("Class sayısı:", num_classes)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D

model = Sequential([
    Conv2D(32, (3,3), activation="relu", padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(64, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(128, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(256, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(), 
    
    Dense(512, activation="relu"),
    BatchNormalization(),
    Dropout(0.5),
    
    Dense(256, activation="relu"),
    Dropout(0.3),

    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "deepfake_multiclass_cnn.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

In [ ]:
pred_probs = model.predict(test_gen)

pred_classes = np.argmax(pred_probs, axis=1)
true_classes = test_gen.classes

class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(true_classes, pred_classes)

plt.figure(figsize=(12,10))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

print(classification_report(true_classes, pred_classes, target_names=class_names))

In [ ]:
def predict_single_image(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)[0]

    index_to_class = {v: k for k, v in train_gen.class_indices.items()}

    predicted_index = np.argmax(predictions)
    predicted_class = index_to_class[predicted_index]
    confidence = predictions[predicted_index]

    print("Tahmin edilen class:", predicted_class)
    print("Güven oranı:", confidence)

    if predicted_class.lower() == "real":
        print("Sonuç: Görsel GERÇEK olarak tahmin edildi.")
    else:
        print("Sonuç: Görsel SAHTE olarak tahmin edildi.")
        print("Tahmin edilen yapay zeka kaynağı:", predicted_class)

    print("\nTüm class olasılıkları:")
    
    for i, prob in enumerate(predictions):
        class_name = index_to_class[i]
        print(f"{class_name}: {prob:.4f}")

In [ ]:
sample_img = test_df.iloc[2000]["filepath"]

print("Gerçek label:", test_df.iloc[2000]["label"])
predict_single_image(sample_img)

In [ ]:
def predict_single_image_with_plot(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)[0]

    index_to_class = {v: k for k, v in train_gen.class_indices.items()}

    class_names = [index_to_class[i] for i in range(len(predictions))]

    predicted_index = np.argmax(predictions)
    predicted_class = class_names[predicted_index]

    plt.figure(figsize=(10,5))
    sns.barplot(x=class_names, y=predictions)
    plt.xticks(rotation=45)
    plt.title("Class Olasılıkları")
    plt.xlabel("Class")
    plt.ylabel("Probability")
    plt.show()

    print("Tahmin:", predicted_class)

    if predicted_class.lower() == "real":
        print("Sonuç: Gerçek görsel")
    else:
        print("Sonuç: Sahte görsel")
        print("AI Kaynağı:", predicted_class)

In [ ]:
predict_single_image_with_plot(sample_img)